## Data Preparation

In this notebook, we're going to download some HuggingFace datasets and then prepare our IFT datasets for our focus languages: German, Japanese, Spanish, Indonesian, and Czech.

In [2]:
import uuid

import pandas as pd
from datasets import Dataset, load_dataset

/Users/ljvmiranda/Developer/multilingual-teacher-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Listing datasets of interest

Let's download the datasets first and see what we have. In this case, let us download the following:
- WildChat 4.8M (allenai/WildChat-4.8M), for `generation` and `respond`. Can be used for most languages, but not evenly distributed.
- GSM8k ([openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k)), for `translation` in Math tasks. Can be used for most focus languages.
- [Magpie-Pro-300K-Filtered](https://huggingface.co/datasets/Magpie-Align/Magpie-Pro-300K-Filtered), for `translation` in General tasks. Can be used for most focus languages.
- [HuggingFaceH4/Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking), for `translation` in General tasks. Filter by language and only get the prompt. Might be more useful for focus languages because it seems that the prompts are culturally-adapted in some way. Only for **German** and **Spanish**
- [nvidia/Helpsteer3](https://huggingface.co/datasets/nvidia/HelpSteer3/viewer/preference/train?f%5Bdomain%5D%5Bvalue%5D=%27multilingual%27&views%5B%5D=preference_train), for `generation` (get the prompt and preferred response) and `respond` in General domains. Not sure what's the full distribution, but might be something to check later on.
- [OpenAssistant/oasst](https://huggingface.co/datasets/OpenAssistant/oasst2): this is quite old, but useful for `respond`. Just filter everything with the prompter role.

Let's look for some additional IFT datasets for Indonesian and Czech (quite mid-resourced):


#### Processing WildChat

In [9]:
# Language mapping: full name to ISO 639-1 code (two-letter)
LANG_MAPPING = {
    "Spanish": "es",
    "German": "de",
    "Indonesian": "id",
    "Czech": "cs",
    "Japanese": "ja",
}

num_instances = 200_000
wildchat_4_8m = load_dataset("allenai/WildChat-4.8M", split="train", streaming=True)
sampled = wildchat_4_8m.shuffle(seed=42).take(num_instances)

sampled_df = pd.DataFrame(list(sampled))
filtered_df = sampled_df[(sampled_df["language"].isin(LANG_MAPPING.keys()))]
print(filtered_df["language"].value_counts())

# Transform to desired format
wildchat_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(filtered_df))],
        "source": "allenai/WildChat-4.8M",
        "conversation": filtered_df["conversation"].values,
        "language": filtered_df["language"].map(LANG_MAPPING).values,
        "strategy": [["generate", "respond"] for _ in range(len(filtered_df))],
        "source_id": filtered_df["conversation_hash"].values,
    }
)

wildchat_df.head()

'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/allenai/WildChat-4.8M/resolve/c827c6df8fcf008219ffaffa4d1dd77491099367/data/train-00061-of-00086.parquet
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 469dd921-a2ad-4656-802c-e75e44752639)')' thrown while requesting GET https://huggingface.co/datasets/allenai/WildChat-4.8M/resolve/c827c6df8fcf008219ffaffa4d1dd77491099367/data/train-00061-of-00086.parquet
Retrying in 2s [Retry 2/5].


language
German        5358
Indonesian    4280
Spanish       4181
Japanese       474
Czech           61
Name: count, dtype: int64


,id,source,conversation,language,strategy,source_id
0,d88d06246f6447828811a82d0f1e42bc,allenai/WildChat-4.8M,"[{'content': '{ “Count”:1, “title”: “海美女(AI美女写...",ja,"[generate, respond]",d84b476b5f4fbe8a4980de5b22848139
1,706668df112e483b8f6892500416e7fa,allenai/WildChat-4.8M,[{'content': 'Gib mir nur 10 Keywords bestehen...,de,"[generate, respond]",7ec811ded963b5b53b18bed926020709
2,a5e65df61d0c407fbca664d9fb66a184,allenai/WildChat-4.8M,[{'content': 'Gib mir nur 10 Keywords bestehen...,de,"[generate, respond]",3bfd2012a08f4781b255214d5db36f88
3,96527d813e4c4d29a70c53b320e7abb0,allenai/WildChat-4.8M,[{'content': 'Gib mir nur 10 Keywords bestehen...,de,"[generate, respond]",18a331c9d7537d743e950298b2066004
4,f3df3ff0e16743249844645ec5cdd88c,allenai/WildChat-4.8M,[{'content': ' 夕方の五時近くになり、別荘の周辺の遊歩道を散策するという女ども...,ja,"[generate, respond]",1415723b869a04ff31edbf7b6b7750cd


NameError: name 'wildchat_df' is not defined

Then let's load the smaller datasets

In [ ]:
# fmt: off
gsm8k = load_dataset("openai/gsm8k", "main", split="train").to_pandas()
magpie_pro_300k = load_dataset("Magpie-Align/Magpie-Pro-300K-Filtered", split="train").to_pandas()
huggingface_h4_multilingual_thinking = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train").to_pandas()
nvidia_helpsteer3 = load_dataset("nvidia/HelpSteer3", "preference", split="train").to_pandas()
oasst2 = load_dataset("OpenAssistant/oasst2", split="train").to_pandas()
# fmt: on